In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

In [ ]:
import gradio as gr
import torch
import ast
import re
import black
import autopep8
from transformers import pipeline
from reportlab.pdfgen import canvas
from datetime import datetime

In [ ]:
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95

In [ ]:
device = 0 if torch.cuda.is_available() else -1

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=device
)

conversation_history = []

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def detect_python_code(text):

    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for pattern in patterns:
        if re.search(pattern, text):
            return True

    return False


In [ ]:
def check_syntax(code):

    try:
        ast.parse(code)
        return True, None

    except SyntaxError as e:
        return False, f"Syntax Error at line {e.lineno}: {e.msg}"

In [ ]:
def fix_code_indentation(code):

    code = code.replace("\t", " ")
    lines = code.split("\n")

    cleaned = [line.rstrip() for line in lines]

    return "\n".join(cleaned)

In [ ]:
def fix_code_brackets(code):

    brackets = {"(": ")", "[": "]", "{": "}"}

    for open_b, close_b in brackets.items():

        if code.count(open_b) > code.count(close_b):

            code += close_b * (code.count(open_b) - code.count(close_b))

    return code

In [ ]:
def format_code(code):

    try:
        formatted = black.format_str(code, mode=black.FileMode())
        return formatted

    except:
        return autopep8.fix_code(code)

In [ ]:
def fix_python_code(code):

    code = fix_code_indentation(code)
    code = fix_code_brackets(code)

    formatted = format_code(code)

    return formatted

In [ ]:
def generate_response(message, temperature, max_tokens, top_p):

    conversation_history.append({"role": "user", "content": message})

    prompt = message

    result = model_pipe(
        prompt,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        top_p=top_p,
        do_sample=True
    )

    reply = result[0]["generated_text"]

    conversation_history.append({"role": "assistant", "content": reply})

    return reply


In [ ]:
def chat(user_message, chat_history, temperature, max_tokens, top_p):

    if detect_python_code(user_message):

        valid, error = check_syntax(user_message)

        if not valid:

            fixed = fix_python_code(user_message)

            reply = f"❌ Syntax Error:\n{error}\n\n✅ Fixed Code:\n{fixed}"

        else:

            reply = "✅ Your Python code syntax looks correct."

    else:

        reply = generate_response(user_message, temperature, max_tokens, top_p)

    chat_history.append((user_message, reply))

    return chat_history, ""


In [ ]:
def download_txt():

    filename = "chat_history.txt"

    with open(filename, "w") as f:

        for msg in conversation_history:

            f.write(f"{msg['role'].upper()}: {msg['content']}\n\n")

    return filename

In [ ]:
def download_pdf():

    filename = "chat_history.pdf"

    c = canvas.Canvas(filename)

    y = 800

    for msg in conversation_history:

        text = f"{msg['role'].upper()}: {msg['content']}"

        c.drawString(50, y, text[:100])

        y -= 20

    c.save()

    return filename


In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 🧠 The Syntax Surgeon")

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(label="Enter message or Python code")

    with gr.Row():

        temperature = gr.Slider(0.1,1,value=0.7,label="Temperature")

        max_tokens = gr.Slider(50,1024,value=256,label="Max Tokens")

        top_p = gr.Slider(0.1,1,value=0.95,label="Top-P")

    send = gr.Button("Send")

    txt_btn = gr.Button("Download TXT")

    pdf_btn = gr.Button("Download PDF")

    send.click(
        chat,
        inputs=[msg, chatbot, temperature, max_tokens, top_p],
        outputs=[chatbot, msg]
    )

    txt_btn.click(download_txt, outputs=gr.File())

    pdf_btn.click(download_pdf, outputs=gr.File())

demo.launch(share=True)


/tmp/ipykernel_291/1483792140.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_291/1483792140.py:5: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c45b14fb60ff716af0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
